In [51]:
!pip install -q -U langchain-google-genai

In [52]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_NEW_KEY')

In [53]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [54]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.0-flash-exp",
    api_key = GEMINI_API_KEY,
    temperature = 0.5
)


In [65]:
from langchain_core.tools import tool
import requests
import json


@tool
def get_exchange_rate(base_currency: int, target_currency: int):
    """
    Retrieves the latest exchange rate between two currencies using an API.

    Args:
        base_currency: The currency to convert from (e.g., "USD").
        target_currency: The currency to convert to (e.g., "EUR").

    Returns:
        A JSON string containing the exchange rate, or an error message.
        Example: '{"rate": 0.85}' or '{"error": "Could not retrieve exchange rate"}'
    """
    try:
        url = f"https://api.exchangerate-api.com/v4/latest/{base_currency}"
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)
        data = response.json()

        if target_currency in data["rates"]:
            rate = data["rates"][target_currency]
            return json.dumps({"rate": rate})
        else:
            return json.dumps({"error": "Currency not supported"})

    except requests.exceptions.RequestException as e:
        return json.dumps({"error": f"Could not retrieve exchange rate: {e}"})
    except (KeyError, TypeError) as e: # Handle potential issues with the JSON structure
        return json.dumps({"error": f"Invalid response from API: {e}"})



@tool
def calculator(num1: int, num2: int):

  """Add, subtract, multiply and divide."""

  num1 = float(input("Enter the first number: "))
  num2 = float(input("Enter the second number: "))

  operation = input("Enter the operation: ")

  if operation == '+':
      result = num1 + num2
  elif operation == '-':
      result = num1 - num2
  elif operation == '*':
      result = num1 * num2
  elif operation == '/':
      if num2 == 0:
          print("Division by 0 is not valid.")
      else:
          result = num1 / num2
  else:
      print("Invalid Entry")
  print("Result: ", result)


# Example usage (testing):
# print(add_numbers("5 + 3"))
# print(add_numbers("10.5 + 2"))
# print(add_numbers("invalid input"))



# Example usage:
#print(get_exchange_rate("USD", "EUR"))
#print(get_exchange_rate("USD", "JPY"))
#print(get_exchange_rate("EUR", "GBP"))
#print(get_exchange_rate("USD", "INVALID")) # Example of an invalid currency code
#print(get_exchange_rate("INVALID", "EUR")) # Example of an invalid base currency code


tools = [get_exchange_rate, calculator]

In [66]:
llm_with_tools = llm.bind_tools(tools)

In [67]:
messages = input("write your question? ")

write your question? what is the exchange rate of 20 USD to EURO?


In [58]:
from langchain.schema import AIMessage

ai_msg = AIMessage(content="")


In [68]:
for tool_call in ai_msg.tool_calls:
            # Map the tool name to the corresponding function
        tool_name = tool_call["function"]["name"]
        selected_tool = {
              "get_exchange_rate": get_exchange_rate,
              "calculator": calculator
        }[tool_call["name"].lower()]
        tool_msg = selected_tool.invoke(tool_call)
        display(tool_msg)
        print(tool_msg)
